# AST Fase 2 — Variante B (backbone ViT + PNGs propios)

**Modelo:** Audio Spectrogram Transformer — backbone ViT pretrained (`MIT/ast-finetuned-audioset-10-10-0.4593`)  
**Entrada:** PNGs propios (log-mel / CQT / PCEN / Gammatone) en escala de grises  
**Filtros:** Log-Mel · CQT · PCEN · Gammatone  
**Duraciones:** 2s · 3s · 4s · 10s  
**Altura:**  
- log-mel / pcen / gammatone → `num_mel_bins=128` (sin padding)  
- cqt → `num_mel_bins=176` (padding negro de 168 → 176)  

**Anchura:** detectada dinámicamente del primer PNG de cada duración, todas las imágenes se right-pad a ese valor  
**Fine-tuning:** últimas 3 capas del encoder + layernorm + cabeza lineal (backbone congelado)  
**Pooling:** token CLS  
**Positional embeddings:** interpolación bilineal 2D desde pretrained (12×101) a nuestra cuadrícula  
**Validación:** Repeated Stratified 5-Fold × 2 repeticiones = **10 folds** (AST es más pesado que CNN-2D)  
**Threshold:** dinámico por fold — barrido F1-macro sobre val interna (15% del train)  

---
**Antes de correr:** `pip install transformers`  
Correr celdas en orden. El loop principal soporta interrupción y reanudación via checkpoint.

In [1]:
# ─── Celda 1: Imports y configuración global ───────────────────────────────
import gc
import re
import json
import time
import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from transformers import ASTConfig, ASTModel

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, auc
)
from sklearn.utils import class_weight

# Forzar CWD a la raiz del proyecto, sin importar desde donde arranque el kernel
import os
PROJECT_ROOT = '/home/ci2dt2-ai/Proyectos/Psiquiatria'
if os.getcwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)
print('CWD:', os.getcwd())

# ══════════════════════════════════════════════════════════════════
CONDITION    = "Depresión"
SPECS_ROOT   = Path("Clasificación Final/Depresión/specs")
RESULTS_ROOT = Path("Depresion/AST_fase2")

# Nombre del checkpoint pretrained en HuggingFace
PRETRAINED_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

# Cuadrícula de patches del modelo pretrained (no cambiar)
# ASTConfig default: num_mel_bins=128, max_length=1024, patch_size=16, stride=10
# freq_patches = (128-16)//10 + 1 = 12
# time_patches = (1024-16)//10 + 1 = 101
OLD_FREQ_PATCHES = 12
OLD_TIME_PATCHES = 101

# CQT: imagen de 168px de alto → padding a 176 para num_mel_bins=176
# (176-16)//10 + 1 = 17 freq patches (número limpio)
FILTER_CONFIGS = [
    {"name": "log-mel",   "img_h": 128, "ast_h": 128},
    {"name": "cqt",       "img_h": 168, "ast_h": 176},
    {"name": "pcen",      "img_h": 128, "ast_h": 128},
    {"name": "gammatone", "img_h": 128, "ast_h": 128},
]

DURATIONS   = [10, 2, 3, 4]
N_SPLITS    = 5
N_REPEATS   = 2          # 10 folds — AST es más pesado que CNN-2D
BATCH_SIZE  = 32         # menor que CNN-2D por el tamaño del modelo
EPOCHS      = 20
PATIENCE    = 4
VAL_FRAC    = 0.15
N_UNFREEZE  = 3          # últimas N capas del encoder a descongelar
LR          = 2e-5       # lr bajo para fine-tuning
THR_GRID    = np.arange(0.05, 0.96, 0.02)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print("Configuracion cargada")

/home/ci2dt2-ai/Proyectos/Psiquiatria/Depresion/Wav2Vec/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CWD: /home/ci2dt2-ai/Proyectos/Psiquiatria
Dispositivo: cuda
  GPU: NVIDIA GeForce RTX 5080
Configuracion cargada


In [2]:
# ─── Celda 2: Utilidades de dataset ───────────────────────────────────────

def build_file_index(root: Path):
    """Devuelve stems, labels y mapa stem → [paths de segmentos]."""
    img_map    = {}
    stem_label = {}
    for cls_str in ["0", "1"]:
        d = root / cls_str
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)
            stem_label[stem] = int(cls_str)
    stems  = sorted(img_map.keys())
    labels = np.array([stem_label[s] for s in stems], dtype=np.int64)
    return stems, labels, img_map


def get_target_width(img_map: dict) -> int:
    """
    Detecta el ancho máximo entre todos los PNGs del directorio.
    Se usa para hacer right-padding a todas las imágenes al mismo ancho.
    """
    max_w = 0
    for paths in img_map.values():
        for p in paths:
            w = Image.open(p).size[0]  # PIL devuelve (W, H)
            if w > max_w:
                max_w = w
            break  # con 1 imagen por stem basta para estimar (todos deberían ser iguales)
    return max_w


def compute_patch_counts(ast_h, target_w, patch_size=16, stride=10):
    """Calcula el número de patches de frecuencia y tiempo para el AST."""
    freq_patches = (ast_h      - patch_size) // stride + 1
    time_patches = (target_w   - patch_size) // stride + 1
    return freq_patches, time_patches


# ── Transforms ──────────────────────────────────────────────────────────────
# Los PNGs son RGB (colormap inferno). Convertimos a escala de grises
# porque el AST espera 1 canal. La información de amplitud se preserva
# en la luminancia (inferno es monótonamente creciente en luminancia).

BASE_TRANSFORM = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),                             # (1, H, W) en [0, 1]
    T.Normalize(mean=[0.5], std=[0.5]),       # → [-1, 1]
])


class ASTSpectrogramDataset(Dataset):
    """
    Dataset de entrenamiento para el AST.
    Salida por muestra: tensor (max_length, num_mel_bins) — formato que
    espera ASTModel como input_values.
    """
    def __init__(self, stems, img_map, ast_h, target_w):
        self.samples  = []
        self.ast_h    = ast_h      # num_mel_bins (altura objetivo con padding)
        self.target_w = target_w   # max_length (ancho objetivo con padding)
        for stem in stems:
            label = int(img_map[stem][0].parent.name)
            for p in img_map[stem]:
                self.samples.append((p, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        tensor = BASE_TRANSFORM(Image.open(path)).squeeze(0)  # (H_real, W_real)

        # Padding en frecuencia (altura) — solo para CQT: 168 → 176
        if tensor.shape[0] < self.ast_h:
            pad_h = self.ast_h - tensor.shape[0]
            tensor = F.pad(tensor, (0, 0, 0, pad_h), value=-1.0)

        # Padding en tiempo (ancho) — segmentos cortos al final de audios
        if tensor.shape[1] < self.target_w:
            pad_w = self.target_w - tensor.shape[1]
            tensor = F.pad(tensor, (0, pad_w), value=-1.0)

        # Transponer: (H, W) → (W, H) = (max_length, num_mel_bins)
        return tensor.T, torch.tensor(label, dtype=torch.long)


class ASTStemDataset(Dataset):
    """Dataset de inferencia — devuelve (tensor, stem_name) para segment voting."""
    def __init__(self, stems, img_map, ast_h, target_w):
        self.samples  = []
        self.ast_h    = ast_h
        self.target_w = target_w
        for stem in stems:
            for p in img_map[stem]:
                self.samples.append((p, stem))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, stem = self.samples[idx]
        tensor = BASE_TRANSFORM(Image.open(path)).squeeze(0)

        if tensor.shape[0] < self.ast_h:
            tensor = F.pad(tensor, (0, 0, 0, self.ast_h - tensor.shape[0]), value=-1.0)
        if tensor.shape[1] < self.target_w:
            tensor = F.pad(tensor, (0, self.target_w - tensor.shape[1]), value=-1.0)

        return tensor.T, stem

print("Clases de dataset definidas")

Clases de dataset definidas


In [3]:
# ─── Celda 3: Modelo ASTClassifier ────────────────────────────────────────
#
# Estrategia:
#   1. Carga el backbone ASTModel pretrained en AudioSet
#   2. Copia pesos del encoder (12 capas transformer) y layernorm
#      → estos no dependen del tamaño de entrada
#   3. Copia el patch embedding (Conv2d 1→768, kernel 16×16)
#      → tampoco depende del tamaño de entrada
#   4. Interpola los positional embeddings de la cuadrícula original
#      (12 freq × 101 time) a nuestra cuadrícula (new_f × new_t)
#      usando interpolación bilineal 2D para preservar la estructura espacial
#   5. Congela todo; descongela las últimas N_UNFREEZE capas + head
#   6. Agrega cabeza lineal: Linear(768→256)→ReLU→Dropout→Linear(256→1)→Sigmoid
#   7. Clasificación usando el token CLS (índice 0 de last_hidden_state)

class ASTClassifier(nn.Module):
    def __init__(self, ast_h, target_w, n_unfreeze=N_UNFREEZE):
        super().__init__()

        freq_patches = (ast_h    - 16) // 10 + 1
        time_patches = (target_w - 16) // 10 + 1
        print(f"  Patches: freq={freq_patches}  time={time_patches}  "
              f"total={freq_patches*time_patches} (+2 tokens especiales)")

        # Config para nuestras dimensiones
        config = ASTConfig(
            num_mel_bins    = ast_h,
            max_length      = target_w,
            hidden_size     = 768,
            num_hidden_layers = 12,
            num_attention_heads = 12,
            intermediate_size = 3072,
            patch_size      = 16,
            frequency_stride = 10,
            time_stride     = 10,
        )

        # Cargar pretrained y crear backbone con nuestro config
        print(f"  Descargando/cargando {PRETRAINED_NAME} ...")
        pretrained = ASTModel.from_pretrained(PRETRAINED_NAME)
        self.ast   = ASTModel(config)

        # Copiar pesos que no dependen del tamaño de entrada
        # FIX transformers 5.x: ASTModel ya no tiene '.encoder', las capas estan en '.layers'
        self.ast.layers.load_state_dict(
            pretrained.layers.state_dict())
        self.ast.layernorm.load_state_dict(
            pretrained.layernorm.state_dict())
        self.ast.embeddings.patch_embeddings.load_state_dict(
            pretrained.embeddings.patch_embeddings.state_dict())
        self.ast.embeddings.cls_token.data = \
            pretrained.embeddings.cls_token.data.clone()
        self.ast.embeddings.distillation_token.data = \
            pretrained.embeddings.distillation_token.data.clone()

        # Interpolar positional embeddings (2D bilineal)
        old_pos = pretrained.embeddings.position_embeddings.data  # (1, 1214, 768)
        new_pos = self._interp_pos_embed(
            old_pos, freq_patches, time_patches,
            OLD_FREQ_PATCHES, OLD_TIME_PATCHES)
        self.ast.embeddings.position_embeddings = nn.Parameter(new_pos)
        print(f"  Pos embed interpolado: {old_pos.shape} → {new_pos.shape}")

        del pretrained
        gc.collect()

        # Cabeza de clasificación (siempre entrenable)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

        # Fine-tuning: congelar todo y luego descongelar selectivamente
        for param in self.parameters():
            param.requires_grad = False

        # Descongelar últimas n_unfreeze capas del encoder
        for layer in self.ast.layers[-n_unfreeze:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Descongelar layernorm
        for param in self.ast.layernorm.parameters():
            param.requires_grad = True

        # Descongelar cabeza
        for param in self.classifier.parameters():
            param.requires_grad = True

        n_total     = sum(p.numel() for p in self.parameters())
        n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Parametros totales     : {n_total:,}")
        print(f"  Parametros entrenables : {n_trainable:,} "
              f"({100*n_trainable/n_total:.1f}%)")

    @staticmethod
    def _interp_pos_embed(old_pos, new_f, new_t, old_f, old_t):
        """
        Interpola los positional embeddings de la cuadrícula pretrained
        (old_f × old_t) a la nueva cuadrícula (new_f × new_t).
        Los tokens especiales CLS y distillation se conservan sin cambios.

        Args:
            old_pos: tensor (1, old_f*old_t + 2, hidden)
        Returns:
            tensor (1, new_f*new_t + 2, hidden)
        """
        special = old_pos[:, :2, :]   # CLS + distillation: (1, 2, 768)
        patches = old_pos[:, 2:, :]   # (1, old_f*old_t, 768)
        hidden  = patches.shape[-1]

        # Reshape a cuadrícula 2D e interpolar
        patches = patches.reshape(1, old_f, old_t, hidden)
        patches = patches.permute(0, 3, 1, 2)               # (1, hidden, old_f, old_t)
        patches = F.interpolate(
            patches.float(),
            size=(new_f, new_t),
            mode='bilinear',
            align_corners=False
        )                                                     # (1, hidden, new_f, new_t)
        patches = patches.permute(0, 2, 3, 1)               # (1, new_f, new_t, hidden)
        patches = patches.reshape(1, new_f * new_t, hidden)  # (1, new_f*new_t, hidden)

        return torch.cat([special, patches], dim=1)

    def forward(self, x):
        # x: (B, max_length, num_mel_bins)
        outputs = self.ast(input_values=x)
        # Token CLS: índice 0 de last_hidden_state
        cls = outputs.last_hidden_state[:, 0, :]  # (B, 768)
        return self.classifier(cls).squeeze(1)    # (B,)

print("Clase ASTClassifier definida")

Clase ASTClassifier definida


In [4]:
# ─── Celda 4: Entrenamiento, inferencia y calibración de threshold ─────────

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds    = model(xb).view(-1)
        sample_w = cw_tensor[yb.long()]
        loss     = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        # Gradient clipping — importante en fine-tuning de transformers
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def predict_stems(model, stems, img_map, ast_h, target_w):
    """Segment voting: promedio de probabilidades por audio."""
    model.eval()
    ds = ASTStemDataset(stems, img_map, ast_h, target_w)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=4, pin_memory=True, persistent_workers=False)
    probs_by_stem = {s: [] for s in stems}
    with torch.no_grad():
        for xb, stem_names in dl:
            xb  = xb.to(DEVICE, non_blocking=True)
            out = model(xb).view(-1).cpu().numpy()
            for prob, stem in zip(out, stem_names):
                probs_by_stem[stem].append(float(prob))
    return {stem: float(np.mean(probs)) for stem, probs in probs_by_stem.items()}


def calibrate_threshold(model, val_stems, val_labels, img_map, ast_h, target_w):
    """Barrido THR_GRID → threshold que maximiza F1-macro en val interna."""
    val_probs = predict_stems(model, val_stems, img_map, ast_h, target_w)
    probs_arr = np.array([val_probs[s] for s in val_stems])
    best_thr, best_f1 = 0.5, -1.0
    for thr in THR_GRID:
        preds = (probs_arr > thr).astype(int)
        score = f1_score(val_labels, preds, average='macro', zero_division=0)
        if score > best_f1:
            best_f1  = score
            best_thr = float(thr)
    return best_thr

print("Funciones de entrenamiento e inferencia definidas")

Funciones de entrenamiento e inferencia definidas


In [5]:
# ─── Celda 5: Checkpoint ───────────────────────────────────────────────────

def ckpt_path(out_dir: Path) -> Path:
    return out_dir / "_checkpoint.json"

def load_checkpoint(out_dir: Path):
    p = ckpt_path(out_dir)
    if p.exists():
        with open(p) as f:
            data = json.load(f)
        total = N_SPLITS * N_REPEATS
        print(f"  [CKPT] Reanudando desde fold {data['folds_done']}/{total}")
        return data
    return {"folds_done": 0, "fold_metrics": []}

def save_checkpoint(out_dir: Path, ckpt: dict):
    with open(ckpt_path(out_dir), "w") as f:
        json.dump(ckpt, f)

print("Funciones de checkpoint definidas")

Funciones de checkpoint definidas


In [6]:
# ─── Celda 6: Reporte y figuras finales ────────────────────────────────────

def save_results(out_dir, filter_name, dur, ckpt, n_splits, n_repeats):
    fold_metrics = ckpt["fold_metrics"]
    total_folds  = n_splits * n_repeats
    model_label  = f"AST-B [{filter_name}] {dur}s"
    fname        = f"ast_{filter_name}_{dur}s"

    keys = ["acc", "prec", "rec", "f1", "f1mac", "auc"]
    metrics_arr = {k: np.array([d[k] for d in fold_metrics]) for k in keys}
    thresholds  = np.array([d["threshold"] for d in fold_metrics])

    # Tabla por fold
    print(f"\n{'='*76}")
    print(f"RESULTADOS — {model_label}")
    print(f"{n_splits}-Fold x {n_repeats} repeticiones = {total_folds} folds")
    print(f"{'='*76}")
    print(f"\n  {'Rep-Fold':10s}  {'Thr':5s}  Acc    Prec   Rec    F1     F1-Mac AUC")
    for d in fold_metrics:
        print(f"  R{d['rep']:02d}-F{d['fold']:02d}     "
              f"{d['threshold']:.2f}   "
              f"{d['acc']:.3f}  {d['prec']:.3f}  {d['rec']:.3f}  "
              f"{d['f1']:.3f}  {d['f1mac']:.3f}  {d['auc']:.3f}")

    print(f"\n  {'─'*66}")
    print(f"  {'MEDIA':10s}  {thresholds.mean():.2f}   "
          + "  ".join(f"{metrics_arr[k].mean():.3f}" for k in keys))
    print(f"  {'± STD':10s}  {thresholds.std():.2f}   "
          + "  ".join(f"{metrics_arr[k].std():.3f}" for k in keys))

    # Figura 1: Boxplot métricas
    labels_m = {"acc": "Accuracy", "prec": "Precision", "rec": "Recall",
                 "f1": "F1 (pos)", "f1mac": "F1-Macro", "auc": "AUC-ROC"}
    fig, axes = plt.subplots(1, 6, figsize=(20, 4))
    fig.suptitle(f"{model_label} — {total_folds} folds", fontsize=11)
    for ax, (k, lbl) in zip(axes, labels_m.items()):
        ax.boxplot(metrics_arr[k], patch_artist=True,
                   boxprops=dict(facecolor='lightsalmon'),
                   medianprops=dict(color='darkred', linewidth=2))
        ax.set_title(lbl, fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.set_xticks([])
        ax.axhline(metrics_arr[k].mean(), color='red', linestyle='--',
                   linewidth=1, label=f"u={metrics_arr[k].mean():.3f}")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(out_dir / f"boxplot_{fname}.png", dpi=150)
    plt.show()
    plt.close()

    # Figura 2: Distribución thresholds
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(thresholds, bins=10, color='salmon', edgecolor='white', alpha=0.8)
    ax.axvline(thresholds.mean(), color='darkred', linestyle='--',
               label=f"u={thresholds.mean():.2f}  sd={thresholds.std():.2f}")
    ax.set_xlabel("Threshold optimo por fold")
    ax.set_ylabel("Frecuencia")
    ax.set_title(f"Thresholds — {model_label}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"thresholds_{fname}.png", dpi=150)
    plt.show()
    plt.close()

    # Figura 3: Confusion matrix promediada
    cms     = np.array([d["cm"] for d in fold_metrics], dtype=float)
    cm_mean = cms.mean(axis=0)
    cm_std  = cms.std(axis=0)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_mean, annot=False, cmap='Reds', ax=ax,
                xticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"],
                yticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"])
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.5,
                    f"{cm_mean[i,j]:.1f}\n+/-{cm_std[i,j]:.1f}",
                    ha='center', va='center', fontsize=12,
                    color='white' if cm_mean[i,j] > cm_mean.max()/2 else 'black')
    ax.set_title(f"Confusion media {total_folds} folds — {model_label}")
    ax.set_ylabel("Real"); ax.set_xlabel("Prediccion")
    plt.tight_layout()
    plt.savefig(out_dir / f"confusion_{fname}.png", dpi=150)
    plt.show()
    plt.close()

    # Confusion matrices y ROC por fold
    cm_dir  = out_dir / "confusion_matrices"
    roc_dir = out_dir / "roc_por_fold"
    cm_dir.mkdir(exist_ok=True)
    roc_dir.mkdir(exist_ok=True)

    for d in fold_metrics:
        rep, fold = d["rep"], d["fold"]
        # CM individual
        cm = np.array(d["cm"])
        np.savetxt(cm_dir / f"cm_R{rep:02d}_F{fold:02d}.csv", cm, delimiter=",", fmt="%d")
        plt.figure(figsize=(4, 3))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
                    xticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"],
                    yticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"])
        plt.title(f"CM R{rep:02d} F{fold:02d}")
        plt.tight_layout()
        plt.savefig(cm_dir / f"cm_R{rep:02d}_F{fold:02d}.png", dpi=120)
        plt.close()
        # ROC individual
        fpr = np.array(d["roc_fpr"])
        tpr = np.array(d["roc_tpr"])
        plt.figure(figsize=(5, 5))
        plt.plot(fpr, tpr, lw=2, label=f"AUC={d['auc']:.3f}")
        plt.plot([0,1],[0,1],'k--')
        plt.title(f"ROC R{rep:02d} F{fold:02d}")
        plt.legend(loc='lower right')
        plt.tight_layout()
        plt.savefig(roc_dir / f"roc_R{rep:02d}_F{fold:02d}.png", dpi=120)
        plt.close()

    # Figura 4: ROC superpuesta con banda media
    fig, ax = plt.subplots(figsize=(8, 7))
    aucs = [d["auc"] for d in fold_metrics]
    for d in fold_metrics:
        ax.plot(np.array(d["roc_fpr"]), np.array(d["roc_tpr"]),
                alpha=0.3, lw=1, color='salmon')
    mean_fpr = np.linspace(0, 1, 100)
    tprs_interp = [np.interp(mean_fpr, d["roc_fpr"], d["roc_tpr"])
                   for d in fold_metrics]
    mean_tpr = np.mean(tprs_interp, axis=0)
    std_tpr  = np.std(tprs_interp, axis=0)
    ax.plot(mean_fpr, mean_tpr, color='darkred', lw=2,
            label=f"Media AUC = {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}")
    ax.fill_between(mean_fpr, mean_tpr-std_tpr, mean_tpr+std_tpr,
                    alpha=0.15, color='darkred', label="+/- 1 std")
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_xlabel("Tasa de Falsos Positivos")
    ax.set_ylabel("Tasa de Verdaderos Positivos")
    ax.set_title(f"ROC por fold — {model_label}")
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(out_dir / f"roc_{fname}.png", dpi=150)
    plt.show()
    plt.close()

    # CSV métricas
    csv_path = out_dir / f"metrics_{fname}.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["rep","fold","threshold",
                                                "acc","prec","rec","f1","f1mac","auc"])
        writer.writeheader()
        writer.writerows([{k: d[k] for k in ["rep","fold","threshold",
                                               "acc","prec","rec","f1","f1mac","auc"]}
                           for d in fold_metrics])
    print(f"  Guardado en : {out_dir}")
    print(f"  CSV metricas: {csv_path}")

print("Funcion save_results definida")

Funcion save_results definida


In [7]:
# ─── Celda 7: LOOP PRINCIPAL — 4 filtros x 4 duraciones ───────────────────
# Si se interrumpe: volver a correr esta celda, el checkpoint retoma.

TOTAL_CONFIGS = len(FILTER_CONFIGS) * len(DURATIONS)
config_n = 0

for filt_cfg in FILTER_CONFIGS:
    filter_name = filt_cfg["name"]
    ast_h       = filt_cfg["ast_h"]   # num_mel_bins (con padding si aplica)

    for dur in DURATIONS:
        config_n   += 1
        image_root  = SPECS_ROOT / filter_name / f"{dur}s"
        out_dir     = RESULTS_ROOT / filter_name / f"{dur}s"
        out_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n{'='*72}")
        print(f"  CONFIG {config_n}/{TOTAL_CONFIGS} — "
              f"filtro={filter_name}  dur={dur}s  ast_h={ast_h}")
        print(f"{'='*72}")

        if not image_root.exists():
            print(f"  [SKIP] No existe: {image_root}")
            continue

        # Cargar índice de imágenes
        file_stems, file_labels, img_map = build_file_index(image_root)
        if len(file_stems) == 0:
            print(f"  [SKIP] Sin imagenes en {image_root}")
            continue

        # Detectar ancho objetivo (max entre todos los PNGs)
        target_w   = get_target_width(img_map)
        f_p, t_p   = compute_patch_counts(ast_h, target_w)
        total_imgs = sum(len(v) for v in img_map.values())

        print(f"\n  Pacientes totales : {len(file_stems)}")
        print(f"  Sin {CONDITION}       : {int((file_labels==0).sum())}")
        print(f"  Con {CONDITION}       : {int((file_labels==1).sum())}")
        print(f"  Segmentos totales : {total_imgs:,} (~{total_imgs/len(file_stems):.1f}/audio)")
        print(f"  Dim AST entrada   : ({target_w}, {ast_h}) = (max_length, num_mel_bins)")
        print(f"  Patches           : freq={f_p}  time={t_p}  total={f_p*t_p}")

        # Checkpoint
        ckpt       = load_checkpoint(out_dir)
        folds_done = ckpt["folds_done"]
        total_folds = N_SPLITS * N_REPEATS

        if folds_done >= total_folds:
            print(f"\n  [SKIP] Ya completado ({folds_done} folds). Regenerando figuras...")
            save_results(out_dir, filter_name, dur, ckpt, N_SPLITS, N_REPEATS)
            continue

        # Cross-validation
        rskf = RepeatedStratifiedKFold(
            n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=42)
        file_stems_arr = np.array(file_stems)

        print(f"\n  CV: {N_SPLITS}-Fold x {N_REPEATS} reps = {total_folds} folds")
        if folds_done > 0:
            print(f"  Reanudando desde fold {folds_done + 1}/{total_folds}")

        for fold_idx, (tr_idx, te_idx) in enumerate(
                rskf.split(file_stems_arr, file_labels)):

            if fold_idx < folds_done:
                continue

            repeat_n    = fold_idx // N_SPLITS + 1
            fold_n      = fold_idx %  N_SPLITS + 1
            fold_global = fold_idx + 1

            print(f"\n  Rep {repeat_n}/{N_REPEATS}  |  "
                  f"Fold {fold_n}/{N_SPLITS}  [{fold_global}/{total_folds}]")

            tr_stems  = list(file_stems_arr[tr_idx])
            te_stems  = list(file_stems_arr[te_idx])
            tr_labels = file_labels[tr_idx]
            te_labels = file_labels[te_idx]

            # Val interna 15% para calibrar threshold
            sss = StratifiedShuffleSplit(
                n_splits=1, test_size=VAL_FRAC, random_state=42 + fold_idx)
            tr_sub_idx, val_idx = next(sss.split(tr_stems, tr_labels))

            val_stems       = [tr_stems[i] for i in val_idx]
            val_labels_fold = tr_labels[val_idx]
            train_stems     = [tr_stems[i] for i in tr_sub_idx]
            train_labels    = tr_labels[tr_sub_idx]

            train_ds = ASTSpectrogramDataset(train_stems, img_map, ast_h, target_w)
            train_dl = DataLoader(
                train_ds, batch_size=BATCH_SIZE, shuffle=True,
                num_workers=4, pin_memory=True, persistent_workers=False)
            print(f"  Train: {len(train_stems)} pac / {len(train_ds):,} segs  "
                  f"| Val: {len(val_stems)} pac  | Test: {len(te_stems)} pac")

            # Class weights
            weights   = class_weight.compute_class_weight(
                            'balanced',
                            classes=np.unique(train_labels),
                            y=train_labels)
            cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

            # Modelo — se descarga el pretrained solo la primera vez,
            # luego queda en cache de HuggingFace
            model     = ASTClassifier(ast_h, target_w).to(DEVICE)
            optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
            criterion = nn.BCELoss(reduction='none')
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                            optimizer, T_max=EPOCHS, eta_min=1e-7)

            # Entrenamiento
            best_loss, patience_cnt, best_state = float('inf'), 0, None
            t0 = time.time()

            for epoch in range(EPOCHS):
                loss = train_one_epoch(
                    model, train_dl, optimizer, criterion, cw_tensor)
                scheduler.step()
                if loss < best_loss:
                    best_loss    = loss
                    patience_cnt = 0
                    best_state   = {k: v.cpu().clone()
                                    for k, v in model.state_dict().items()}
                    mark = "v"
                else:
                    patience_cnt += 1
                    mark = f"({patience_cnt}/{PATIENCE})"
                print(f"  Ep {epoch+1:02d}/{EPOCHS}  "
                      f"loss {loss:.4f}  best {best_loss:.4f}  {mark}")
                if patience_cnt >= PATIENCE:
                    print("  Early stopping.")
                    break

            model.load_state_dict(best_state)
            print(f"  Entrenamiento: {time.time()-t0:.1f}s")

            # Calibrar threshold
            best_thr = calibrate_threshold(
                model, val_stems, val_labels_fold, img_map, ast_h, target_w)
            print(f"  Threshold optimo: {best_thr:.2f}")

            # Evaluar en test
            stem_probs = predict_stems(model, te_stems, img_map, ast_h, target_w)
            fold_true, fold_pred, fold_prob = [], [], []
            for stem, lbl in zip(te_stems, te_labels):
                prob = stem_probs[stem]
                fold_true.append(int(lbl))
                fold_pred.append(1 if prob > best_thr else 0)
                fold_prob.append(prob)

            fold_true_arr = np.array(fold_true)
            fold_prob_arr = np.array(fold_prob)

            if len(np.unique(fold_true_arr)) > 1:
                fpr, tpr, _ = roc_curve(fold_true_arr, fold_prob_arr)
                fold_auc    = float(auc(fpr, tpr))
                fpr_list    = fpr.tolist()
                tpr_list    = tpr.tolist()
            else:
                fold_auc = 0.0
                fpr_list = [0.0, 1.0]
                tpr_list = [0.0, 1.0]
                print("  [AVISO] Test de este fold tiene una sola clase — AUC=0")

            cm_fold = confusion_matrix(fold_true, fold_pred, labels=[0,1]).tolist()

            m = {
                "rep"      : repeat_n,
                "fold"     : fold_n,
                "threshold": round(best_thr, 4),
                "acc"      : float(accuracy_score(fold_true, fold_pred)),
                "prec"     : float(precision_score(fold_true, fold_pred, zero_division=0)),
                "rec"      : float(recall_score(fold_true, fold_pred, zero_division=0)),
                "f1"       : float(f1_score(fold_true, fold_pred, zero_division=0)),
                "f1mac"    : float(f1_score(fold_true, fold_pred,
                                            average='macro', zero_division=0)),
                "auc"      : fold_auc,
                "cm"       : cm_fold,
                "roc_fpr"  : fpr_list,
                "roc_tpr"  : tpr_list,
            }
            print(f"  acc={m['acc']:.3f}  prec={m['prec']:.3f}  "
                  f"rec={m['rec']:.3f}  F1={m['f1']:.3f}  "
                  f"F1-mac={m['f1mac']:.3f}  AUC={m['auc']:.3f}  "
                  f"thr={m['threshold']:.2f}")

            # Checkpoint
            ckpt["fold_metrics"].append(m)
            ckpt["folds_done"] = fold_global
            save_checkpoint(out_dir, ckpt)

            # Liberar memoria
            del model, train_ds, train_dl, best_state, cw_tensor
            gc.collect()
            torch.cuda.empty_cache()

        save_results(out_dir, filter_name, dur, ckpt, N_SPLITS, N_REPEATS)

print(f"\n{'='*72}")
print(f"  FASE 2 — AST Variante B COMPLETADA — resultados en: {RESULTS_ROOT}")
print(f"{'='*72}")


  CONFIG 1/16 — filtro=log-mel  dur=10s  ast_h=128

  Pacientes totales : 79
  Sin Depresión       : 58
  Con Depresión       : 21
  Segmentos totales : 1,259 (~15.9/audio)
  Dim AST entrada   : (313, 128) = (max_length, num_mel_bins)
  Patches           : freq=12  time=30  total=360

  CV: 5-Fold x 2 reps = 10 folds

  Rep 1/2  |  Fold 1/5  [1/10]
  Train: 53 pac / 799 segs  | Val: 10 pac  | Test: 16 pac
  Patches: freq=12  time=30  total=360 (+2 tokens especiales)
  Descargando/cargando MIT/ast-finetuned-audioset-10-10-0.4593 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15532.15it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Pos embed interpolado: torch.Size([1, 1214, 768]) → torch.Size([1, 362, 768])
  Parametros totales     : 85,730,049
  Parametros entrenables : 21,462,273 (25.0%)
  Ep 01/20  loss 0.6678  best 0.6678  v
  Ep 02/20  loss 0.5623  best 0.5623  v
  Ep 03/20  loss 0.4708  best 0.4708  v
  Ep 04/20  loss 0.3496  best 0.3496  v
  Ep 05/20  loss 0.2178  best 0.2178  v
  Ep 06/20  loss 0.1117  best 0.1117  v
  Ep 07/20  loss 0.0511  best 0.0511  v
  Ep 08/20  loss 0.0193  best 0.0193  v
  Ep 09/20  loss 0.0051  best 0.0051  v
  Ep 10/20  loss 0.0024  best 0.0024  v
  Ep 11/20  loss 0.0016  best 0.0016  v
  Ep 12/20  loss 0.0015  best 0.0015  v
  Ep 13/20  loss 0.0012  best 0.0012  v
  Ep 14/20  loss 0.0010  best 0.0010  v
  Ep 15/20  loss 0.0009  best 0.0009  v
  Ep 16/20  loss 0.0008  best 0.0008  v
  Ep 17/20  loss 0.0008  best 0.0008  (1/4)
  Ep 18/20  loss 0.0008  best 0.0008  (2/4)
  Ep 19/20  loss 0.0008  best 0.0008  (3/4)
  Ep 20/20  loss 0.0008  best 0.0008  (4/4)
  Early stopping.
  

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18501.27it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Pos embed interpolado: torch.Size([1, 1214, 768]) → torch.Size([1, 362, 768])
  Parametros totales     : 85,730,049
  Parametros entrenables : 21,462,273 (25.0%)
  Ep 01/20  loss 0.6826  best 0.6826  v
  Ep 02/20  loss 0.5700  best 0.5700  v
  Ep 03/20  loss 0.4961  best 0.4961  v
  Ep 04/20  loss 0.3453  best 0.3453  v
  Ep 05/20  loss 0.2045  best 0.2045  v
  Ep 06/20  loss 0.1120  best 0.1120  v
  Ep 07/20  loss 0.0472  best 0.0472  v
  Ep 08/20  loss 0.0339  best 0.0339  v
  Ep 09/20  loss 0.0169  best 0.0169  v
  Ep 10/20  loss 0.0045  best 0.0045  v
  Ep 11/20  loss 0.0019  best 0.0019  v
  Ep 12/20  loss 0.0015  best 0.0015  v
  Ep 13/20  loss 0.0014  best 0.0014  v
  Ep 14/20  loss 0.0011  best 0.0011  v
  Ep 15/20  loss 0.0011  best 0.0011  (1/4)
  Ep 16/20  loss 0.0011  best 0.0011  v
  Ep 17/20  loss 0.0010  best 0.0010  v
  Ep 18/20  loss 0.0009  best 0.0009  v
  Ep 19/20  loss 0.0009  best 0.0009  (1/4)
  Ep 20/20  loss 0.0010  best 0.0009  (2/4)
  Entrenamiento: 56.9s
 

...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21438.54it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Pos embed interpolado: torch.Size([1, 1214, 768]) → torch.Size([1, 146, 768])
  Parametros totales     : 85,564,161
  Parametros entrenables : 21,462,273 (25.1%)
  Ep 01/20  loss 0.5920  best 0.5920  v
  Ep 02/20  loss 0.4458  best 0.4458  v
  Ep 03/20  loss 0.3563  best 0.3563  v
  Ep 04/20  loss 0.2734  best 0.2734  v
  Ep 05/20  loss 0.2061  best 0.2061  v
  Ep 06/20  loss 0.1500  best 0.1500  v
  Ep 07/20  loss 0.0822  best 0.0822  v
  Ep 08/20  loss 0.0568  best 0.0568  v
  Ep 09/20  loss 0.0205  best 0.0205  v
  Ep 10/20  loss 0.0228  best 0.0205  (1/4)
  Ep 11/20  loss 0.0073  best 0.0073  v
  Ep 12/20  loss 0.0012  best 0.0012  v
  Ep 13/20  loss 0.0004  best 0.0004  v
  Ep 14/20  loss 0.0002  best 0.0002  v
  Ep 15/20  loss 0.0001  best 0.0001  v
  Ep 16/20  loss 0.0001  best 0.0001  v
  Ep 17/20  loss 0.0001  best 0.0001  (1/4)
  Ep 18/20  loss 0.0001  best 0.0001  v
  Ep 19/20  loss 0.0001  best 0.0001  v
  Ep 20/20  loss 0.0001  best 0.0001  v
  Entrenamiento: 104.5s
  Th

/tmp/ipykernel_3667811/449814150.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_3667811/449814150.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_3667811/449814150.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Guardado en : Depresion/AST_fase2/gammatone/4s
  CSV metricas: Depresion/AST_fase2/gammatone/4s/metrics_ast_gammatone_4s.csv

  FASE 2 — AST Variante B COMPLETADA — resultados en: Depresion/AST_fase2


/tmp/ipykernel_3667811/449814150.py:139: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Para graficar bonitas las graficas de Confusion Matrix y ROC curve
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import auc
from pathlib import Path

def graficar_resultados_bonitos(ckpt, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics = ckpt["fold_metrics"]

    # =================================================================
    # 1. MATRIZ DE CONFUSIÓN GLOBAL (Suma de todos los Folds)
    # =================================================================
    cm_total = np.zeros((2, 2), dtype=int)
    for m in metrics:
        cm_total += np.array(m['cm']) # Sumamos las predicciones

    plt.figure(figsize=(6, 5))
    # Usamos formato 'd' para enteros y un estilo limpio
    sns.heatmap(cm_total, annot=True, fmt='d', cmap='Blues', cbar=False,
                annot_kws={"size": 15}, linewidths=1, linecolor='black')

    plt.title("Matriz de Confusión Global (Suma de Folds)", fontsize=14, pad=15)
    plt.ylabel('Etiqueta Real', fontsize=12, fontweight='bold')
    plt.xlabel('Predicción del Modelo', fontsize=12, fontweight='bold')
    plt.xticks([0.5, 1.5], [f'0 (Sin {CONDITION})', f'1 (Con {CONDITION})'], fontsize=11)
    plt.yticks([0.5, 1.5], [f'0 (Sin {CONDITION})', f'1 (Con {CONDITION})'], fontsize=11, va='center')

    plt.tight_layout()
    plt.savefig(out_dir / "cm_global_bonita.png", dpi=300, bbox_inches='tight')
    plt.close()

    # =================================================================
    # 2. CURVA ROC PROMEDIO CON DESVIACIÓN ESTÁNDAR SOMBREADA
    # =================================================================
    tprs = []
    aucs = []
    mean_fpr = np.linspace(0, 1, 100) # Base común para poder promediar

    plt.figure(figsize=(8, 6))

    for m in metrics:
        fpr_fold = m['roc_fpr']
        tpr_fold = m['roc_tpr']

        # Interpolar la curva actual al eje común (mean_fpr)
        interp_tpr = np.interp(mean_fpr, fpr_fold, tpr_fold)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        aucs.append(m['auc'])

        # Plotear la línea del fold individual, pero MUY clarita al fondo (opcional)
        plt.plot(fpr_fold, tpr_fold, color='gray', alpha=0.1, lw=1)

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)

    # Sombreado de la desviación estándar (+/- 1 std)
    std_tpr = np.std(tprs, axis=0)
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1) # Asegurar que no pase de 1.0
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0) # Asegurar que no baje de 0.0

    plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='#1f77b4', alpha=0.2,
                     label=r'$\pm$ 1 Desv. Estándar')

    # Plot de la línea media
    plt.plot(mean_fpr, mean_tpr, color='#1f77b4', lw=2.5,
             label=f'ROC Promedio (AUC = {mean_auc:.3f} $\pm$ {std_auc:.3f})')

    # Línea de referencia (Azar)
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='tomato', label='Azar', alpha=0.8)

    plt.xlim([-0.02, 1.02])
    plt.ylim([-0.02, 1.02])
    plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12, fontweight='bold')
    plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12, fontweight='bold')
    plt.title(f'Curva ROC Global con Validación Cruzada', fontsize=14, pad=15)
    plt.legend(loc="lower right", fontsize=11, framealpha=0.9)
    plt.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.savefig(out_dir / "roc_global_bonita.png", dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✅ ¡Gráficas generadas exitosamente en {out_dir}!")

<>:72: SyntaxWarning: invalid escape sequence '\p'
<>:72: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_3667811/2860080969.py:72: SyntaxWarning: invalid escape sequence '\p'
  label=f'ROC Promedio (AUC = {mean_auc:.3f} $\pm$ {std_auc:.3f})')


In [9]:
import json
from pathlib import Path

# Celda opcional: regenera las graficas "bonitas" (matriz de confusion global + ROC
# promedio) para UNA config ya completada. Edita filtro/duracion abajo segun cual
# quieras graficar (correr esto DESPUES de que el loop principal - celda 7 - haya
# terminado esa config). RESULTS_ROOT ya apunta a la condicion correcta (Depresion),
# no hace falta tocar nada mas que filtro/duracion.
filtro   = "gammatone"   # cambia esto por el filtro que quieras graficar
duracion = "2s"          # cambia esto por la duracion que quieras graficar

ruta_resultados = RESULTS_ROOT / filtro / duracion
archivo_json = ruta_resultados / "_checkpoint.json"

if archivo_json.exists():
    with open(archivo_json) as f:
        ckpt = json.load(f)
    graficar_resultados_bonitos(ckpt, ruta_resultados)
else:
    print(f"No se encontro el checkpoint: {archivo_json}")


✅ ¡Gráficas generadas exitosamente en Depresion/AST_fase2/gammatone/2s!
